# RL Market Making on QBTS MBO Data

This notebook implements:
1. **Data preprocessing** - Convert raw MBO events to 1-second order book snapshots
2. **Avellaneda-Stoikov baseline** - Analytical optimal quoting model
3. **DQN agent** - Deep Q-Network for learning quote placement
4. **PPO agent** - Proximal Policy Optimization for continuous improvement
5. **Comparison** - Evaluate all approaches on held-out test data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_maker.data_preprocessor import preprocess_all, load_snapshots
from market_maker.env import MarketMakingEnv
from market_maker.baseline import AvellanedaStoikov, run_baseline, grid_search_baseline

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Preprocess MBO Data

This converts each day's raw MBO events (~2M events/day) into 1-second snapshots
with order book features. Only needs to run once results are cached to parquet.

In [ ]:
DATA_DIR = "XNAS-20260227-QVD7UYV7GQ"
SNAPSHOT_DIR = "snapshots"

preprocess_all(DATA_DIR, SNAPSHOT_DIR, freq="1s")

In [ ]:
train_data, test_data = load_snapshots(SNAPSHOT_DIR)

print(f"\nTrain columns: {train_data.columns.tolist()}")
print(f"\nSample snapshot:")
train_data.head()

In [ ]:
# Quick sanity check: plot mid price and features for one day
day1 = train_data[train_data["timestamp"].dt.date == train_data["timestamp"].dt.date.iloc[0]]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(day1["timestamp"], day1["mid_price"])
axes[0].set_ylabel("Mid Price ($)")
axes[0].set_title(f"QBTS — {day1['timestamp'].dt.date.iloc[0]}")

axes[1].plot(day1["timestamp"], day1["spread"], color="orange")
axes[1].set_ylabel("Spread ($)")

axes[2].plot(day1["timestamp"], day1["book_imbalance"], color="green")
axes[2].set_ylabel("Book Imbalance")
axes[2].axhline(0, color="gray", ls="--", lw=0.5)

axes[3].plot(day1["timestamp"], day1["ofi_rolling"], color="purple")
axes[3].set_ylabel("OFI (rolling)")
axes[3].axhline(0, color="gray", ls="--", lw=0.5)

plt.tight_layout()
plt.show()

## Step 2: Avellaneda-Stoikov Baseline

The classical analytical solution for optimal market making.
We grid-search over gamma (risk aversion) and k (order arrival intensity)
to find the best parameters, then use this as our benchmark.

In [ ]:
# Create environment with test data
test_env = MarketMakingEnv(
    data=test_data,
    max_inventory=1000,
    inventory_penalty=0.01,
    fill_size=100,
    episode_length=3600,  # 1 hour episodes
)

print(f"Action space: {test_env.action_space}")
print(f"Observation space: {test_env.observation_space}")

# Quick test
obs, info = test_env.reset()
print(f"\nInitial observation: {obs}")
print(f"Observation shape: {obs.shape}")

In [ ]:
# Grid search for best AS parameters
as_results = grid_search_baseline(
    test_env,
    gammas=[0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    ks=[0.5, 1.0, 1.5, 2.0, 5.0, 10.0],
)

print("\nTop 5 parameter combinations:")
as_results.sort_values("total_pnl", ascending=False).head()

In [ ]:
# Run best AS config and record trajectory for plotting
best_row = as_results.sort_values("total_pnl", ascending=False).iloc[0]
best_gamma, best_k = best_row["gamma"], best_row["k"]
print(f"Best AS params: gamma={best_gamma}, k={best_k}")

as_model = AvellanedaStoikov(gamma=best_gamma, k=best_k)

# Run one episode and collect trajectory
obs, _ = test_env.reset()
done = False
as_pnl_trace = []
as_inv_trace = []

while not done:
    mid = test_env.data.iloc[test_env._step_idx]["mid_price"]
    sigma = test_env.data.iloc[test_env._step_idx]["volatility"]
    sigma = sigma if not np.isnan(sigma) and sigma > 0 else 0.001
    t_rem = 1.0 - obs[-1]
    
    action = as_model.get_action_for_env(
        mid, test_env.inventory, sigma, t_rem,
        test_env.BID_OFFSETS, test_env.ASK_OFFSETS
    )
    obs, reward, terminated, truncated, info = test_env.step(action)
    as_pnl_trace.append(info["total_pnl"])
    as_inv_trace.append(info["inventory"])
    done = terminated or truncated

as_stats = test_env.get_episode_stats()
print(f"\nAS Baseline Results:")
for k, v in as_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 3: Train DQN Agent

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

# Create training environment
train_env = Monitor(MarketMakingEnv(
    data=train_data,
    max_inventory=1000,
    inventory_penalty=0.01,
    fill_size=100,
    episode_length=3600,
))

dqn_model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    target_update_interval=500,
    policy_kwargs={"net_arch": [128, 128]},
    verbose=1,
)

print("Training DQN...")
dqn_model.learn(total_timesteps=100_000)
dqn_model.save("models/dqn_market_maker")

In [ ]:
# Evaluate DQN on test data
dqn_eval_env = MarketMakingEnv(
    data=test_data,
    max_inventory=1000,
    inventory_penalty=0.01,
    fill_size=100,
    episode_length=3600,
)

obs, _ = dqn_eval_env.reset()
done = False
dqn_pnl_trace = []
dqn_inv_trace = []

while not done:
    action, _ = dqn_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = dqn_eval_env.step(action)
    dqn_pnl_trace.append(info["total_pnl"])
    dqn_inv_trace.append(info["inventory"])
    done = terminated or truncated

dqn_stats = dqn_eval_env.get_episode_stats()
print("DQN Results:")
for k, v in dqn_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 4: Train PPO Agent

In [ ]:
from stable_baselines3 import PPO

ppo_train_env = Monitor(MarketMakingEnv(
    data=train_data,
    max_inventory=1000,
    inventory_penalty=0.01,
    fill_size=100,
    episode_length=3600,
))

ppo_model = PPO(
    "MlpPolicy",
    ppo_train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs={"net_arch": [128, 128]},
    verbose=1,
)

ppo_model.learn(total_timesteps=200_000)
ppo_model.save("models/ppo_market_maker")

In [ ]:
# Evaluate PPO on test data
ppo_eval_env = MarketMakingEnv(
    data=test_data,
    max_inventory=1000,
    inventory_penalty=0.01,
    fill_size=100,
    episode_length=3600,
)

obs, _ = ppo_eval_env.reset()
done = False
ppo_pnl_trace = []
ppo_inv_trace = []

while not done:
    action, _ = ppo_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = ppo_eval_env.step(action)
    ppo_pnl_trace.append(info["total_pnl"])
    ppo_inv_trace.append(info["inventory"])
    done = terminated or truncated

ppo_stats = ppo_eval_env.get_episode_stats()
print("PPO Results:")
for k, v in ppo_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Step 5: Compare All Approaches

In [ ]:
comparison = pd.DataFrame([
    {"Agent": "Avellaneda-Stoikov", **as_stats},
    {"Agent": "DQN", **dqn_stats},
    {"Agent": "PPO", **ppo_stats},
])

print("\n" + "="*60)
print("AGENT COMPARISON")
print("="*60)
comparison[["Agent", "total_pnl", "sharpe", "max_drawdown", "n_trades", "final_inventory", "max_inventory"]]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# PnL traces
axes[0].plot(as_pnl_trace, label="Avellaneda-Stoikov", alpha=0.8)
axes[0].plot(dqn_pnl_trace, label="DQN", alpha=0.8)
axes[0].plot(ppo_pnl_trace, label="PPO", alpha=0.8)
axes[0].set_ylabel("Cumulative PnL ($)")
axes[0].set_title("Market Making Agent Comparison — Test Episode")
axes[0].legend()
axes[0].axhline(0, color="gray", ls="--", lw=0.5)

# Inventory traces
axes[1].plot(as_inv_trace, label="Avellaneda-Stoikov", alpha=0.8)
axes[1].plot(dqn_inv_trace, label="DQN", alpha=0.8)
axes[1].plot(ppo_inv_trace, label="PPO", alpha=0.8)
axes[1].set_ylabel("Inventory (shares)")
axes[1].set_xlabel("Time Step (seconds)")
axes[1].legend()
axes[1].axhline(0, color="gray", ls="--", lw=0.5)

plt.tight_layout()
plt.savefig("agent_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from collections import Counter

def collect_actions(model_or_fn, env, n_steps=3600):
    obs, _ = env.reset()
    actions = []
    for _ in range(n_steps):
        if callable(model_or_fn):
            action = model_or_fn(obs, env)
        else:
            action, _ = model_or_fn.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        actions.append(int(action))
        if terminated or truncated:
            break
    return actions

dqn_actions = collect_actions(dqn_model, MarketMakingEnv(test_data, episode_length=3600))
ppo_actions = collect_actions(ppo_model, MarketMakingEnv(test_data, episode_length=3600))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(*zip(*sorted(Counter(dqn_actions).items())))
axes[0].set_xlabel("Action")
axes[0].set_ylabel("Count")
axes[0].set_title("DQN Action Distribution")

axes[1].bar(*zip(*sorted(Counter(ppo_actions).items())))
axes[1].set_xlabel("Action")
axes[1].set_ylabel("Count")
axes[1].set_title("PPO Action Distribution")

plt.tight_layout()
plt.show()